# Cyberbullying Detection — Intelex vs CuPy benchmark

Dataset: https://www.kaggle.com/datasets/sayankr007/cyber-bullying-data-for-multi-label-classification?select=final_hateXplain.csv

Models: LightGBM, XGBoost, Logistic Regression, Random Forest, AdaBoost, MLP.

Sections are grouped by implementation because `patch_sklearn()` patches the whole kernel: run **BareBones** and **CuPy** first, then the single patch cell, then **Intelex**.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report
from lightgbm import LGBMClassifier
from cupyx.scipy.sparse import csr_matrix
import warnings
import cupy as cp
from sklearnex import patch_sklearn
import numpy as np
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.neural_network import MLPClassifier


## BareBones (plain scikit-learn)

### LightGBM

In [ ]:
#Suppress FutureWarnings
warnings.filterwarnings("ignore", category=FutureWarning)


def mainBareBones():
    # Load the dataset
    data = pd.read_csv('final_hateXplain.csv', dtype={
        'comment': 'string',
        'label': 'category',
        'Race': 'category',
        'Religion': 'category',
        'Gender': 'category',
        'Sexual Orientation': 'category',
        'Miscellaneous': 'category'
    })

    # Preprocessing
    # Fill missing values in the 'comment' column
    data['comment'] = data['comment'].fillna('')

    # Encode the 'label' column (e.g., normal, offensive, hatespeech)
    label_encoder = LabelEncoder()
    data['label'] = label_encoder.fit_transform(data['label'])

    # TF-IDF Vectorization for the 'comment' column
    tfidf = TfidfVectorizer(max_features=5000)
    X = tfidf.fit_transform(data['comment'])

    # Target variable
    y = data['label']

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # LightGBM Classifier
    model = LGBMClassifier(
        objective='multiclass',
        num_class=len(label_encoder.classes_),
        random_state=42,
        force_col_wise=True
    )
    model.fit(X_train, y_train)

    # Predictions on test data
    y_pred = model.predict(X_test)

    # Evaluation
    print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))


if __name__ == "__main__":
    mainBareBones()


[LightGBM] [Info] Total Bins 64010
[LightGBM] [Info] Number of data points in the train set: 16087, number of used features: 1775
[LightGBM] [Info] Start training from score -1.137269
[LightGBM] [Info] Start training from score -0.934925
[LightGBM] [Info] Start training from score -1.249350
              precision    recall  f1-score   support

  hatespeech       0.75      0.69      0.72      1325
      normal       0.62      0.80      0.70      1502
   offensive       0.58      0.42      0.49      1195

    accuracy                           0.65      4022
   macro avg       0.65      0.63      0.63      4022
weighted avg       0.65      0.65      0.64      4022



### XGBoost

In [ ]:
def mainBareBones():
    # Load the dataset
    data = pd.read_csv('final_hateXplain.csv', dtype={
        'comment': 'string',  
        'label': 'category', 
        'Race': 'category',
        'Religion': 'category',
        'Gender': 'category',
        'Sexual Orientation': 'category',
        'Miscellaneous': 'category'
    })

    # Preprocessing
    # Fill missing values in the 'comment' column
    data['comment'] = data['comment'].fillna('')

    # Encode the 'label' column (e.g., normal, offensive, hatespeech)
    label_encoder = LabelEncoder()
    data['label'] = label_encoder.fit_transform(data['label'])

    # TF-IDF Vectorization for the 'comment' column
    tfidf = TfidfVectorizer(max_features=5000) 
    X = tfidf.fit_transform(data['comment'])

    # Target variable
    y = data['label']

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # XGBoost Model
    model = XGBClassifier(eval_metric='mlogloss')
    model.fit(X_train, y_train)

    # Predictions on test data
    y_pred = model.predict(X_test)

    # Evaluation
    print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

if __name__ == "__main__":
    mainBareBones()

              precision    recall  f1-score   support

  hatespeech       0.75      0.66      0.71      1325
      normal       0.60      0.81      0.69      1502
   offensive       0.56      0.39      0.46      1195

    accuracy                           0.64      4022
   macro avg       0.64      0.62      0.62      4022
weighted avg       0.64      0.64      0.63      4022



### Logistic Regression

In [ ]:
def mainBareBones():
    # Load the dataset
    data = pd.read_csv('final_hateXplain.csv', dtype={
        'comment': 'string',  
        'label': 'category', 
        'Race': 'category',
        'Religion': 'category',
        'Gender': 'category',
        'Sexual Orientation': 'category',
        'Miscellaneous': 'category'
    })

    # Preprocessing
    # Fill missing values in the 'comment' column
    data['comment'] = data['comment'].fillna('')

    # Encode the 'label' column (e.g., normal, offensive, hatespeech)
    label_encoder = LabelEncoder()
    data['label'] = label_encoder.fit_transform(data['label'])

    # TF-IDF Vectorization for the 'comment' column
    tfidf = TfidfVectorizer(max_features=5000)
    X = tfidf.fit_transform(data['comment'])

    # Target variable
    y = data['label']

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Logistic Regression Model
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)

    # Predictions on test data
    y_pred = model.predict(X_test)

    # Evaluation
    print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

if __name__ == "__main__":
    mainBareBones()


              precision    recall  f1-score   support

  hatespeech       0.72      0.68      0.70      1325
      normal       0.61      0.77      0.68      1502
   offensive       0.55      0.41      0.47      1195

    accuracy                           0.63      4022
   macro avg       0.63      0.62      0.61      4022
weighted avg       0.63      0.63      0.62      4022



### Random Forest

In [ ]:
def mainBareBones():
    # Load the dataset
    data = pd.read_csv('final_hateXplain.csv', dtype={
        'comment': 'string',  
        'label': 'category', 
        'Race': 'category',
        'Religion': 'category',
        'Gender': 'category',
        'Sexual Orientation': 'category',
        'Miscellaneous': 'category'
    })

    # Preprocessing
    # Fill missing values in the 'comment' column
    data['comment'] = data['comment'].fillna('')

    # Encode the 'label' column (e.g., normal, offensive, hatespeech)
    label_encoder = LabelEncoder()
    data['label'] = label_encoder.fit_transform(data['label'])

    # TF-IDF Vectorization for the 'comment' column
    tfidf = TfidfVectorizer(max_features=5000)
    X = tfidf.fit_transform(data['comment'])

    # Target variable
    y = data['label']

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Random Forest Classifier
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    # Predictions on test data
    y_pred = model.predict(X_test)

    # Evaluation
    print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

if __name__ == "__main__":
    mainBareBones()


              precision    recall  f1-score   support

  hatespeech       0.72      0.66      0.69      1325
      normal       0.56      0.84      0.67      1502
   offensive       0.60      0.29      0.39      1195

    accuracy                           0.62      4022
   macro avg       0.63      0.60      0.58      4022
weighted avg       0.63      0.62      0.59      4022



### AdaBoost

In [ ]:
def mainBareBones():
    # Load the dataset
    data = pd.read_csv('final_hateXplain.csv', dtype={
        'comment': 'string',  
        'label': 'category', 
        'Race': 'category',
        'Religion': 'category',
        'Gender': 'category',
        'Sexual Orientation': 'category',
        'Miscellaneous': 'category'
    })

    # Preprocessing
    # Fill missing values in the 'comment' column
    data['comment'] = data['comment'].fillna('')

    # Encode the 'label' column (e.g., normal, offensive, hatespeech)
    label_encoder = LabelEncoder()
    data['label'] = label_encoder.fit_transform(data['label'])

    # TF-IDF Vectorization for the 'comment' column
    tfidf = TfidfVectorizer(max_features=5000) 
    X = tfidf.fit_transform(data['comment'])

    # Target variable
    y = data['label']

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # AdaBoost Model
    print("\nInitializing AdaBoost Classifier...")
    model = AdaBoostClassifier(n_estimators=50, random_state=42)

    # Train the model
    print("\nTraining the AdaBoost model...")
    model.fit(X_train, y_train)
    print("Model training completed!")

    # Predictions on test data
    print("\nMaking predictions on the test set...")
    y_pred = model.predict(X_test)

    # Evaluation
    print("\nEvaluating the model...")
    print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

if __name__ == "__main__":
    mainBareBones()


### MLP

In [ ]:
def mainBareBones():
    # Suppress warnings
    warnings.filterwarnings("ignore", category=FutureWarning)

    # Load the dataset
    data = pd.read_csv('final_hateXplain.csv', dtype={
        'comment': 'string',  
        'label': 'category', 
        'Race': 'category',
        'Religion': 'category',
        'Gender': 'category',
        'Sexual Orientation': 'category',
        'Miscellaneous': 'category'
    })

    # Preprocessing
    # Fill missing values in the 'comment' column
    data['comment'] = data['comment'].fillna('')

    # Encode the 'label' column (e.g., normal, offensive, hatespeech)
    label_encoder = LabelEncoder()
    data['label'] = label_encoder.fit_transform(data['label'])

    # TF-IDF Vectorization for the 'comment' column
    tfidf = TfidfVectorizer(max_features=5000) 
    X = tfidf.fit_transform(data['comment'])

    # Target variable
    y = data['label']

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # MLP Model
    print("\nTraining the MLP model...")
    model = MLPClassifier(
        hidden_layer_sizes=(50, 30),  
        max_iter=1200,               
        learning_rate='adaptive',            
        early_stopping=True,        
        random_state=42              
    )
    model.fit(X_train, y_train)
    print("Model training completed!")

    # Predictions on test data
    print("\nMaking predictions on the test set...")
    y_pred = model.predict(X_test)

    # Evaluation
    print("\nEvaluating the model...")
    print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

if __name__ == "__main__":
    mainBareBones()


Training the MLP model...
Model training completed!

Making predictions on the test set...

Evaluating the model...
              precision    recall  f1-score   support

  hatespeech       0.66      0.70      0.68      1325
      normal       0.62      0.72      0.67      1502
   offensive       0.53      0.38      0.44      1195

    accuracy                           0.61      4022
   macro avg       0.60      0.60      0.60      4022
weighted avg       0.61      0.61      0.60      4022



## CuPy

### LightGBM

In [ ]:
def optimize_datatypes(data):
    for col in data.select_dtypes(include=['int']).columns:
        data[col] = cp.asnumpy(cp.array(data[col], dtype=cp.int32))
    for col in data.select_dtypes(include=['float']).columns:
        data[col] = cp.asnumpy(cp.array(data[col], dtype=cp.float32))
    return data


def mainCuPY():
    # Load the dataset
    data = pd.read_csv('final_hateXplain.csv', dtype={
        'comment': 'string',
        'label': 'category',
        'Race': 'category',
        'Religion': 'category',
        'Gender': 'category',
        'Sexual Orientation': 'category',
        'Miscellaneous': 'category'
    })

    # Preprocessing
    # Fill missing values in the 'comment' column
    data['comment'] = data['comment'].fillna('')

    # Encode the 'label' column (e.g., normal, offensive, hatespeech)
    label_encoder = LabelEncoder()
    data['label'] = label_encoder.fit_transform(data['label'])

    # Optimize datatypes
    data = optimize_datatypes(data)

    # TF-IDF Vectorization for the 'comment' column
    tfidf = TfidfVectorizer(max_features=5000)
    X_csr = tfidf.fit_transform(data['comment'])
    X = csr_matrix(X_csr)

    # Target variable
    y = cp.array(data['label'])

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # LightGBM Classifier
    model = LGBMClassifier(
        objective='multiclass',
        num_class=len(label_encoder.classes_),
        random_state=42,
        force_col_wise=True
    )
    model.fit(cp.asnumpy(X_train.toarray()), cp.asnumpy(y_train))

    # Predictions on test data
    y_pred = model.predict(cp.asnumpy(X_test.toarray()))

    # Evaluation
    print(classification_report(cp.asnumpy(y_test), y_pred, target_names=label_encoder.classes_))


if __name__ == "__main__":
    mainCuPY()

[LightGBM] [Info] Total Bins 64010
[LightGBM] [Info] Number of data points in the train set: 16087, number of used features: 1775
[LightGBM] [Info] Start training from score -1.137269
[LightGBM] [Info] Start training from score -0.934925
[LightGBM] [Info] Start training from score -1.249350
              precision    recall  f1-score   support

  hatespeech       0.75      0.69      0.72      1325
      normal       0.62      0.80      0.70      1502
   offensive       0.58      0.42      0.49      1195

    accuracy                           0.65      4022
   macro avg       0.65      0.63      0.63      4022
weighted avg       0.65      0.65      0.64      4022



### XGBoost

In [ ]:
def mainCuPY():
    # Load the dataset with optimized datatypes
    data = pd.read_csv('final_hateXplain.csv', dtype={
        'comment': 'string',  
        'label': 'category', 
        'Race': 'category',
        'Religion': 'category',
        'Gender': 'category',
        'Sexual Orientation': 'category',
        'Miscellaneous': 'category'
    })

    # Fill missing values in the 'comment' column
    data['comment'] = data['comment'].fillna('')

    # Encode the 'label' column with LabelEncoder
    label_encoder = LabelEncoder()
    data['label'] = label_encoder.fit_transform(data['label']).astype('int8')

    # TF-IDF Vectorization
    tfidf = TfidfVectorizer(max_features=5000)
    X = tfidf.fit_transform(data['comment'])

    # Convert the sparse matrix to CuPy format
    X_cupy = cp.sparse.csr_matrix(X, dtype=cp.float32)

    y = cp.asarray(data['label'], dtype=cp.int32)

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X_cupy, y, test_size=0.2, random_state=42
    )

    # XGBoost parameters
    params = {
        'objective': 'binary:logistic',
        'tree_method': 'hist',  # Optimized for GPU
        'device': 'cuda',     # Use GPU
        'random_state': 42
    }

    # XGBoost Model
    model = XGBClassifier(**params)
    model.fit(X_train.get(), y_train.get())  # Convert CuPy arrays to NumPy

    # Predictions
    y_pred = model.predict(X_test.get())

    # Evaluation
    print(classification_report(y_test.get(), y_pred, target_names=label_encoder.classes_))

if __name__ == "__main__":
    mainCuPY()


              precision    recall  f1-score   support

  hatespeech       0.25      0.01      0.02      1325
      normal       0.39      0.91      0.55      1502
   offensive       0.15      0.07      0.09      1195

    accuracy                           0.36      4022
   macro avg       0.26      0.33      0.22      4022
weighted avg       0.27      0.36      0.24      4022



C:\Users\antho\anaconda3\envs\sklearn-env\Lib\site-packages\xgboost\core.py:158: UserWarning: [18:43:21] WARNING: D:\bld\xgboost-split_1738880170463\work\src\common\error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  warnings.warn(smsg, UserWarning)


### Logistic Regression

In [ ]:
def mainCuPY():
    # Load the dataset
    data = pd.read_csv('final_hateXplain.csv', dtype={
        'comment': 'string',  
        'label': 'category', 
        'Race': 'category',
        'Religion': 'category',
        'Gender': 'category',
        'Sexual Orientation': 'category',
        'Miscellaneous': 'category'
    })

    # Preprocessing
    # Fill missing values in the 'comment' column
    data['comment'] = data['comment'].fillna('')

    # Encode the 'label' column (e.g., normal, offensive, hatespeech)
    label_encoder = LabelEncoder()
    data['label'] = label_encoder.fit_transform(data['label'])

    tfidf = TfidfVectorizer(max_features=5000) 
    X = tfidf.fit_transform(data['comment'])

   
    X = cp.sparse.csr_matrix(X)

    # Target variable
    y = cp.asarray(data['label'])

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    model = LogisticRegression(max_iter=1000)  
    model.fit(X_train.get(), y_train.get())

    # Predictions on test data
    y_pred = model.predict(X_test.get())

    # Evaluation
    print(classification_report(y_test.get(), y_pred, target_names=label_encoder.classes_))

if __name__ == "__main__":
    mainCuPY()


              precision    recall  f1-score   support

  hatespeech       0.72      0.68      0.70      1325
      normal       0.61      0.77      0.68      1502
   offensive       0.55      0.41      0.47      1195

    accuracy                           0.63      4022
   macro avg       0.63      0.62      0.61      4022
weighted avg       0.63      0.63      0.62      4022



### Random Forest

In [ ]:
def mainCuPY():
    # Load the dataset
    data = pd.read_csv('final_hateXplain.csv', dtype={
        'comment': 'string',  
        'label': 'category', 
        'Race': 'category',
        'Religion': 'category',
        'Gender': 'category',
        'Sexual Orientation': 'category',
        'Miscellaneous': 'category'
    })

    # Preprocessing
    # Fill missing values in the 'comment' column
    data['comment'] = data['comment'].fillna('')

    # Encode the 'label' column (e.g., normal, offensive, hatespeech)
    label_encoder = LabelEncoder()
    data['label'] = label_encoder.fit_transform(data['label'])

    # TF-IDF Vectorization for the 'comment' column
    tfidf = TfidfVectorizer(max_features=5000)
    X = tfidf.fit_transform(data['comment'])

    # Convert sparse matrix to CuPy
    X = cp.sparse.csr_matrix(X)

    # Target variable
    y = cp.asarray(data['label'])

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train.get(), y_train.get())

    # Predictions on test data
    y_pred = model.predict(X_test.get())

    # Evaluation
    print(classification_report(y_test.get(), y_pred, target_names=label_encoder.classes_))

if __name__ == "__main__":
    mainCuPY()


              precision    recall  f1-score   support

  hatespeech       0.72      0.66      0.69      1325
      normal       0.56      0.84      0.67      1502
   offensive       0.60      0.29      0.39      1195

    accuracy                           0.62      4022
   macro avg       0.63      0.60      0.58      4022
weighted avg       0.63      0.62      0.59      4022



### AdaBoost

In [ ]:
def mainCuPY():
    # Load the dataset
    data = pd.read_csv('final_hateXplain.csv', dtype={
        'comment': 'string',  
        'label': 'category', 
        'Race': 'category',
        'Religion': 'category',
        'Gender': 'category',
        'Sexual Orientation': 'category',
        'Miscellaneous': 'category'
    })

    # Preprocessing
    # Fill missing values in the 'comment' column
    data['comment'] = data['comment'].fillna('')

    # Encode the 'label' column (e.g., normal, offensive, hatespeech)
    label_encoder = LabelEncoder()
    data['label'] = label_encoder.fit_transform(data['label'])

    tfidf = TfidfVectorizer(max_features=5000) 
    X = tfidf.fit_transform(data['comment'])

    # Move TF-IDF matrix to CuPy
    X = cp.array(X.toarray())

    # Target variable
    y = cp.array(data['label'])  # Move labels to GPU

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print(f"Training set shape: {X_train.shape}, Test set shape: {X_test.shape}")

    print("\nInitializing AdaBoost Classifier...")
    model = AdaBoostClassifier(n_estimators=50, random_state=42)

    # Train the model 
    print("\nTraining the AdaBoost model...")
    model.fit(cp.asnumpy(X_train), cp.asnumpy(y_train))  # Convert CuPy arrays
    print("Model training completed!")

    # Predictions on test data
    print("\nMaking predictions on the test set...")
    y_pred = model.predict(cp.asnumpy(X_test))

    # Evaluation
    print("\nEvaluating the model...")
    print(classification_report(cp.asnumpy(y_test), y_pred, target_names=label_encoder.classes_))

if __name__ == "__main__":
    mainCuPY()


### MLP

In [ ]:
def mainCuPY():
    # Suppress warnings
    warnings.filterwarnings("ignore", category=FutureWarning)

    # Load the dataset
    data = pd.read_csv('final_hateXplain.csv', dtype={
        'comment': 'string',  
        'label': 'category', 
        'Race': 'category',
        'Religion': 'category',
        'Gender': 'category',
        'Sexual Orientation': 'category',
        'Miscellaneous': 'category'
    })

    # Preprocessing
    # Fill missing values in the 'comment' column
    data['comment'] = data['comment'].fillna('')

    # Encode the 'label' column (e.g., normal, offensive, hatespeech)
    label_encoder = LabelEncoder()
    data['label'] = label_encoder.fit_transform(data['label'])

    tfidf = TfidfVectorizer(max_features=5000)
    X = tfidf.fit_transform(data['comment'])
    X = cp.array(X.toarray(), dtype=cp.float32)  # Convert to CuPy array

    # Target variable
    y = cp.array(data['label'].values, dtype=cp.int32)

    # Train-test split
    print("\nSplitting data into training and testing sets...")
    X_train, X_test, y_train, y_test = train_test_split(
        X.get(), y.get(), test_size=0.2, random_state=42
    )  # Convert back to NumPy
    print(f"Training set shape: {X_train.shape}, Testing set shape: {X_test.shape}")

    # MLP Model
    print("\nTraining the MLP model...")
    model = MLPClassifier(
        hidden_layer_sizes=(50, 30),  
        max_iter=1200,               
        learning_rate='adaptive',            
        early_stopping=True,        
        random_state=42              
    )
    model.fit(X_train, y_train)
    print("Model training completed!")

    # Predictions on test data
    print("\nMaking predictions on the test set...")
    y_pred = model.predict(X_test)

    # Evaluation
    print("\nEvaluating the model...")
    print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

if __name__ == "__main__":
    mainCuPY()



Vectorizing text data with TF-IDF...

Splitting data into training and testing sets...
Training set shape: (16087, 5000), Testing set shape: (4022, 5000)

Training the MLP model...
Model training completed!

Making predictions on the test set...

Evaluating the model...
              precision    recall  f1-score   support

  hatespeech       0.66      0.70      0.68      1325
      normal       0.62      0.72      0.67      1502
   offensive       0.53      0.38      0.44      1195

    accuracy                           0.61      4022
   macro avg       0.60      0.60      0.60      4022
weighted avg       0.61      0.61      0.60      4022



## Intelex (scikit-learn-intelex)

Apply the Intel patch once; everything below runs patched.

In [ ]:
patch_sklearn()  # Apply Intel optimizations


Intel(R) Extension for Scikit-learn* enabled (https://github.com/intel/scikit-learn-intelex)


### LightGBM

In [ ]:
def mainIntelex():
    # Load the dataset
    data = pd.read_csv('final_hateXplain.csv', dtype={
        'comment': 'string',
        'label': 'category',
        'Race': 'category',
        'Religion': 'category',
        'Gender': 'category',
        'Sexual Orientation': 'category',
        'Miscellaneous': 'category'
    })

    # Preprocessing
    # Fill missing values in the 'comment' column
    data['comment'] = data['comment'].fillna('')

    # Encode the 'label' column (e.g., normal, offensive, hatespeech)
    label_encoder = LabelEncoder()
    data['label'] = label_encoder.fit_transform(data['label'])

    # TF-IDF Vectorization for the 'comment' column
    tfidf = TfidfVectorizer(max_features=5000)
    X = tfidf.fit_transform(data['comment'])

    # Target variable
    y = data['label']

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # LightGBM Classifier
    model = LGBMClassifier(
        objective='multiclass',
        num_class=len(label_encoder.classes_),
        random_state=42,
        force_col_wise=True
    )
    model.fit(X_train, y_train)

    # Predictions on test data
    y_pred = model.predict(X_test)

    # Evaluation
    print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))


if __name__ == "__main__":
    mainIntelex()

[LightGBM] [Info] Total Bins 64010
[LightGBM] [Info] Number of data points in the train set: 16087, number of used features: 1775
[LightGBM] [Info] Start training from score -1.137269
[LightGBM] [Info] Start training from score -0.934925
[LightGBM] [Info] Start training from score -1.249350
              precision    recall  f1-score   support

  hatespeech       0.75      0.69      0.72      1325
      normal       0.62      0.80      0.70      1502
   offensive       0.58      0.42      0.49      1195

    accuracy                           0.65      4022
   macro avg       0.65      0.63      0.63      4022
weighted avg       0.65      0.65      0.64      4022



### XGBoost

In [ ]:
def mainIntelex():
    # Load the dataset
    data = pd.read_csv('final_hateXplain.csv', dtype={
        'comment': 'string',  
        'label': 'category', 
        'Race': 'category',
        'Religion': 'category',
        'Gender': 'category',
        'Sexual Orientation': 'category',
        'Miscellaneous': 'category'
    })

    # Preprocessing
    # Fill missing values in the 'comment' column
    data['comment'] = data['comment'].fillna('')

    # Encode the 'label' column (e.g., normal, offensive, hatespeech)
    label_encoder = LabelEncoder()
    data['label'] = label_encoder.fit_transform(data['label'])

    # TF-IDF Vectorization for the 'comment' column
    tfidf = TfidfVectorizer(max_features=5000)  # Adjust max_features as needed
    X = tfidf.fit_transform(data['comment'])

    # Target variable
    y = data['label']

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # XGBoost Model
    model = XGBClassifier(eval_metric='mlogloss')
    model.fit(X_train, y_train)

    # Predictions on test data
    y_pred = model.predict(X_test)

    # Evaluation
    print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

if __name__ == "__main__":
    mainIntelex()

              precision    recall  f1-score   support

  hatespeech       0.75      0.66      0.71      1325
      normal       0.60      0.81      0.69      1502
   offensive       0.56      0.39      0.46      1195

    accuracy                           0.64      4022
   macro avg       0.64      0.62      0.62      4022
weighted avg       0.64      0.64      0.63      4022



### Logistic Regression

In [ ]:
def mainIntelex():
    # Load the dataset
    data = pd.read_csv('final_hateXplain.csv', dtype={
        'comment': 'string',  
        'label': 'category', 
        'Race': 'category',
        'Religion': 'category',
        'Gender': 'category',
        'Sexual Orientation': 'category',
        'Miscellaneous': 'category'
    })

    # Preprocessing
    # Fill missing values in the 'comment' column
    data['comment'] = data['comment'].fillna('')

    # Encode the 'label' column (e.g., normal, offensive, hatespeech)
    label_encoder = LabelEncoder()
    data['label'] = label_encoder.fit_transform(data['label'])

    # TF-IDF Vectorization for the 'comment' column
    tfidf = TfidfVectorizer(max_features=5000)
    X = tfidf.fit_transform(data['comment'])

    # Target variable
    y = data['label']

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Logistic Regression Model
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)

    # Predictions on test data
    y_pred = model.predict(X_test)

    # Evaluation
    print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

if __name__ == "__main__":
    mainIntelex()


              precision    recall  f1-score   support

  hatespeech       0.72      0.68      0.70      1325
      normal       0.61      0.77      0.68      1502
   offensive       0.55      0.41      0.47      1195

    accuracy                           0.63      4022
   macro avg       0.63      0.62      0.61      4022
weighted avg       0.63      0.63      0.62      4022



### Random Forest

In [ ]:
def mainIntelex():
    # Load the dataset
    data = pd.read_csv('final_hateXplain.csv', dtype={
        'comment': 'string',  
        'label': 'category', 
        'Race': 'category',
        'Religion': 'category',
        'Gender': 'category',
        'Sexual Orientation': 'category',
        'Miscellaneous': 'category'
    })

    # Preprocessing
    # Fill missing values in the 'comment' column
    data['comment'] = data['comment'].fillna('')

    # Encode the 'label' column (e.g., normal, offensive, hatespeech)
    label_encoder = LabelEncoder()
    data['label'] = label_encoder.fit_transform(data['label'])

    # TF-IDF Vectorization for the 'comment' column
    tfidf = TfidfVectorizer(max_features=5000)
    X = tfidf.fit_transform(data['comment'])

    # Target variable
    y = data['label']

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Random Forest Classifier
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    # Predictions on test data
    y_pred = model.predict(X_test)

    # Evaluation
    print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

if __name__ == "__main__":
    mainIntelex()


              precision    recall  f1-score   support

  hatespeech       0.72      0.66      0.69      1325
      normal       0.56      0.84      0.67      1502
   offensive       0.60      0.29      0.39      1195

    accuracy                           0.62      4022
   macro avg       0.63      0.60      0.58      4022
weighted avg       0.63      0.62      0.59      4022



### AdaBoost

In [ ]:
def mainIntelex():
    # Load the dataset
    data = pd.read_csv('final_hateXplain.csv', dtype={
        'comment': 'string',  
        'label': 'category', 
        'Race': 'category',
        'Religion': 'category',
        'Gender': 'category',
        'Sexual Orientation': 'category',
        'Miscellaneous': 'category'
    })

    # Preprocessing
    # Fill missing values in the 'comment' column
    data['comment'] = data['comment'].fillna('')

    # Encode the 'label' column (e.g., normal, offensive, hatespeech)
    label_encoder = LabelEncoder()
    data['label'] = label_encoder.fit_transform(data['label'])

    # TF-IDF Vectorization for the 'comment' column
    tfidf = TfidfVectorizer(max_features=5000) 
    X = tfidf.fit_transform(data['comment'])

    # Target variable
    y = data['label']

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # AdaBoost Model
    print("\nInitializing AdaBoost Classifier...")
    model = AdaBoostClassifier(n_estimators=50, random_state=42)

    # Train the model
    print("\nTraining the AdaBoost model...")
    model.fit(X_train, y_train)
    print("Model training completed!")

    # Predictions on test data
    print("\nMaking predictions on the test set...")
    y_pred = model.predict(X_test)

    # Evaluation
    print("\nEvaluating the model...")
    print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

if __name__ == "__main__":
    mainIntelex()


### MLP

In [ ]:
def mainIntelex():
    # Suppress warnings
    warnings.filterwarnings("ignore", category=FutureWarning)

    # Load the dataset
    data = pd.read_csv('final_hateXplain.csv', dtype={
        'comment': 'string',  
        'label': 'category', 
        'Race': 'category',
        'Religion': 'category',
        'Gender': 'category',
        'Sexual Orientation': 'category',
        'Miscellaneous': 'category'
    })

    # Preprocessing
    # Fill missing values in the 'comment' column
    data['comment'] = data['comment'].fillna('')

    # Encode the 'label' column (e.g., normal, offensive, hatespeech)
    label_encoder = LabelEncoder()
    data['label'] = label_encoder.fit_transform(data['label'])

    # TF-IDF Vectorization for the 'comment' column
    tfidf = TfidfVectorizer(max_features=5000) 
    X = tfidf.fit_transform(data['comment'])

    # Target variable
    y = data['label']

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # MLP Model
    print("\nTraining the MLP model...")
    model = MLPClassifier(
        hidden_layer_sizes=(50, 30),  
        max_iter=1200,               
        learning_rate='adaptive',            
        early_stopping=True,        
        random_state=42              
    )
    model.fit(X_train, y_train)
    print("Model training completed!")

    # Predictions on test data
    print("\nMaking predictions on the test set...")
    y_pred = model.predict(X_test)

    # Evaluation
    print("\nEvaluating the model...")
    print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

if __name__ == "__main__":
    mainIntelex()


Training the MLP model...
Model training completed!

Making predictions on the test set...

Evaluating the model...
              precision    recall  f1-score   support

  hatespeech       0.66      0.70      0.68      1325
      normal       0.62      0.72      0.67      1502
   offensive       0.53      0.38      0.44      1195

    accuracy                           0.61      4022
   macro avg       0.60      0.60      0.60      4022
weighted avg       0.61      0.61      0.60      4022

